[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/55_qlora_solution.ipynb)

# 🔴 Solution: QLoRA Linear Layer

Reference solution for QLoRA: a quantized base weight combined with low-rank adaptation.

$$W_{\text{eff}} = \text{dequant}(W_q) + \frac{\alpha}{r} \cdot A B$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class QLoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.quantized_weight = nn.Parameter(
            torch.zeros(out_features, in_features, dtype=torch.int8), requires_grad=False)
        self.scale = nn.Parameter(torch.ones(out_features, 1))
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

    def set_weight(self, W_fp32):
        scale = W_fp32.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8) / 127.0
        q = (W_fp32 / scale).round().clamp(-127, 127).to(torch.int8)
        self.quantized_weight.data.copy_(q)
        self.scale.data.copy_(scale)

    def forward(self, x):
        W_fp = self.quantized_weight.float() * self.scale
        base = x @ W_fp.T
        delta = x @ self.lora_A @ self.lora_B * (self.alpha / self.rank)
        return base + delta

In [ ]:
torch.manual_seed(0)
in_f, out_f, rank = 64, 32, 4
layer = QLoRALinear(in_f, out_f, rank, alpha=2.0)

# Set weight from a full-precision reference
W_ref = torch.randn(out_f, in_f)
layer.set_weight(W_ref)

x = torch.randn(8, in_f)
y_qlora = layer(x)
y_ref = x @ W_ref.T  # full-precision baseline (no LoRA delta)

print("Output shape:", y_qlora.shape)          # (8, 32)
print("Max abs error vs fp32:", (y_qlora - y_ref).abs().max().item())

In [ ]:
from torch_judge import check
check("qlora")